<a href="https://colab.research.google.com/github/Amazeble/chatterbox-finetuning/blob/main/colab_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook guides you through fine-tuning the Chatterbox TTS model.

## Step 1: Clone Repository & Install Dependencies

In [ ]:
import os

# Clone the repository if not already present
if not os.path.exists("chatterbox-finetuning"):
    !git clone https://github.com/amazeble/chatterbox-finetuning.git
    %cd chatterbox-finetuning
else:
    %cd chatterbox-finetuning

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install FFmpeg
!apt-get update -qq
!apt-get install -y ffmpeg

# Install Python dependencies
!pip install chatterbox-tts --no-deps
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 torchcodec --index-url https://download.pytorch.org/whl/cu124 --force-reinstall
!pip install peft==0.17.1  --no-deps
!pip install -q -r requirements.txt



## Step 2: Configure Training Settings

In [ ]:
#@title ⚙️ STEP 0: Configuration
#@markdown ### 📂 Project Settings
project_name = "Adriene" #@param {type:"string"}
input_file_path = "/content/drive/MyDrive/Chatterbox/Adriene.wav" #@param {type:"string"}

# --- Paths ---
model_dir = "./pretrained_models"
base_dataset_dir = "./MyTTSDataset"
base_output_dir = "./chatterbox_output"

# --- Dataset Format ---
preprocess = True  # Preprocessing mode: True (always run), False (skip), "auto" (smart detection)

# --- Training Mode ---
is_turbo = True  # True for Turbo training, False for Normal
is_lora = True  # True for LoRA training (efficient, <10h data), False for Full Fine-Tune
is_merge_lora = False  # Auto merge LoRA adapter after training if is_lora=True

# --- Hyperparameters ---
batch_size = 8  # Adjust based on VRAM (2, 4, 8)
grad_accum = 4  # Effective Batch Size = Batch * Accum
learning_rate = 1e-4 if is_lora else 1e-5
num_epochs = 10 if is_lora else 30
save_steps = 500
save_total_limit = 5
dataloader_num_workers = 8

# Save config override for train.py to read
import json
config_override = {
    'project_name': project_name,
    'input_file_path': input_file_path,
    'preprocess': preprocess,
    'is_turbo': is_turbo,
    'is_lora': is_lora,
    'is_merge_lora': is_merge_lora,
    'batch_size': batch_size,
    'grad_accum': grad_accum,
    'learning_rate': learning_rate,
    'num_epochs': num_epochs,
    'save_steps': save_steps,
    'save_total_limit': save_total_limit,
    'dataloader_num_workers': dataloader_num_workers
}
with open('colab_config_override.json', 'w') as f:
    json.dump(config_override, f)
print(f'✅ Config saved: project_name="{project_name}", input_file_path="{input_file_path}"')


In [ ]:
%cd chatterbox-finetuning/

## Step 3: Download and Prepare Base Models

In [ ]:
# Clean pretrained_models if switching modes
if os.path.exists("pretrained_models"):
    import shutil
    shutil.rmtree("pretrained_models")
    print("🗑️ Cleaned old pretrained_models directory")

# Run setup.py
print("\n⏳ Downloading and preparing models...\n")
!python setup.py

print("\n✅ Setup complete! Check the output above for the new_vocab_size value.")

## Step 4: Start Training

In [ ]:
!python train.py --project_name "$project_name" --input_file_path "$input_file_path"

##Test Inference

In [ ]:
#@title 🗣️ Test Speech Synthesis (Inference)
#@markdown Generate speech using your trained model. Make sure you have a reference audio file.

import os
from IPython.display import Audio, display

# Check for speaker reference
ref_dir = "speaker_reference"
if not os.path.exists(ref_dir):
    os.makedirs(ref_dir)
    print(f"Created {ref_dir}/ directory")
    print("\n⚠️ Please upload a reference audio file (3-10 seconds clean speech)")
    print(f"   Upload it to: {ref_dir}/reference.wav")
else:
    ref_files = [f for f in os.listdir(ref_dir) if f.endswith('.wav')]
    if ref_files:
        print(f"Found reference files: {ref_files}")

        # Update inference.py with test text
        test_text = "Hello, this is a test of my custom voice model."  #@param {type:"string"}
        ref_file = ref_files[0]  #@param {type:"string"}

        # Modify inference.py temporarily
        with open('inference.py', 'r') as f:
            inf_content = f.read()

        # Replace test text
        inf_content = inf_content.replace(
            'TEXT_TO_SAY = "Merhaba, sesimi geliştirmem oldukça uzun zaman aldı ve şimdi sahip olduğuma göre, sessiz kalmayacağım."',
            f'TEXT_TO_SAY = "{test_text}"'
        )

        with open('inference.py', 'w') as f:
            f.write(inf_content)

        print(f"\nRunning inference with:")
        print(f"  Text: {test_text}")
        print(f"  Reference: {ref_dir}/{ref_file}")
        print("\nGenerating speech...\n")

        # Run inference
        !python inference.py

        # Play output
        if os.path.exists("output.wav"):
            print("\n✅ Generated output.wav")
            display(Audio("output.wav"))
        else:
            print("\n❌ Output file not generated. Check error messages above.")
    else:
        print(f"\n⚠️ No .wav files found in {ref_dir}/")
        print("Please upload a reference audio file (3-10 seconds of clean speech)")

##Merge LoRA Adapter (Optional)

In [ ]:
#@title 📦 Merge LoRA Adapter into Base Model
#@markdown If you used LoRA training and are satisfied with the results, merge the adapter into a standalone model file.

if is_lora:
    print("Merging LoRA adapter into base model...")
    print("This creates a single .safetensors file ready for deployment.\n")

    !python merge_lora.py

    print("\n✅ Merge complete!")
    merged_model = f"chatterbox_output/{project_name}/t3_turbo_finetuned_merged.safetensors" if is_turbo else f"chatterbox_output/{project_name}/t3_finetuned_merged.safetensors"
    print(f"Merged model saved to: {merged_model}")

    if os.path.exists(merged_model):
        size_mb = os.path.getsize(merged_model) / (1024 * 1024)
        print(f"File size: {size_mb:.1f} MB")
else:
    print("ℹ️ Skipping merge - you used full fine-tuning, not LoRA")

##Download Trained Model

In [ ]:
#@title 💾 Download Trained Model
#@markdown Download your trained model to Google Drive or local storage.

from google.colab import drive
import os
import shutil

# Mount Google Drive
drive.mount('/content/drive', force_remount=False)

# Determine what to save
output_dir = f"chatterbox_output/{project_name}"

if os.path.exists(output_dir):
    # Create backup directory in Drive
    drive_backup = f"/content/drive/MyDrive/chatterbox_models/{project_name}"
    os.makedirs(drive_backup, exist_ok=True)

    # Copy all output files
    for item in os.listdir(output_dir):
        src = os.path.join(output_dir, item)
        dst = os.path.join(drive_backup, item)

        if os.path.isfile(src):
            shutil.copy2(src, dst)
            print(f"✅ Copied: {item} ({os.path.getsize(src) / (1024*1024):.1f} MB)")
        elif os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f"✅ Copied directory: {item}/")

    print(f"\n📁 All files saved to: {drive_backup}")
    print("\nYou can also download individual files using the file browser on the left.")
else:
    print(f"❌ Output directory not found: {output_dir}")
    print("Make sure training has completed successfully.")

---
## Troubleshooting Tips

### Common Issues:

1. **CUDA Out of Memory**
   - Enable LoRA training (`is_lora = True`)
   - Reduce `batch_size` in `src/config.py`
   - Use a smaller dataset

2. **Poor Quality Output**
   - Ensure reference audio is clean (3-10 seconds)
   - Check dataset quality (minimal background noise)
   - Train for more epochs if dataset is small
   - For Turbo model: switch to LoRA if experiencing instability

3. **Vocabulary Mismatch Error**
   - Make sure `new_vocab_size` matches the tokenizer
   - Re-run `setup.py` after changing modes

4. **Missing Reference Audio**
   - Upload a clean 3-10 second WAV file to `speaker_reference/`
   - Use the same speaker as your training dataset for best results

---
**Need Help?** Check the README.md file or open an issue on GitHub.